In [1]:
# cell 1
# Install runtime dependencies for GPT-OSS + vLLM on Colab / Blackwell.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy accelerate safetensors huggingface_hub

# Remove optional packages that may break imports or pull mismatched CUDA wheels.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is useful for tokenizer checks and compatibility.
!uv pip install --system -U "transformers>=4.56.0"

# GPT-OSS support is now in current vLLM releases.
# For this Colab GPU, your previous successful environment was CUDA 13 / Blackwell,
# so first try the cu130 path. If it fails, fall back to vLLM auto backend.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130 || \
 uv pip install --system -U vllm --torch-backend=auto

# Optional fallback only if the lines above fail in your Colab:
# !uv pip install --system --pre -U "vllm==0.10.1+gptoss" \
#     --extra-index-url https://wheels.vllm.ai/gpt-oss/ \
#     --extra-index-url https://download.pytorch.org/whl/nightly/cu128 \
#     --index-strategy unsafe-best-match

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 113ms
Prepared 8 packages in 0.36ms
Uninstalled 8 packages in 119ms
Installed 8 packages in 112ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 47ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 93ms
Checked 27 packages in 0.27ms
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 2.05s
Prepared 10 packages in 19ms
Uninstalled 8 packages in 108ms
Installed 10 packages in 116ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cublas=

In [2]:
# cell 2
# Imports and global config.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback
import site
import glob

from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI
from jsonschema import validate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATASET = "hotpotqa"

# GPT-OSS model.
LLM_MODEL_NAME = "openai/gpt-oss-120b"

# Requested reasoning level:
# GPT-OSS supports low / medium / high; medium is balanced speed and detail.
REASONING_EFFORT = "high"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# GPT-OSS supports 131072 context; keep the same long-context budget.
MAX_MODEL_LEN = 131072

# 95 GB VRAM: use high utilization, but leave a little headroom.
# If startup OOM happens, lower this to 0.92.
GPU_MEMORY_UTILIZATION = 0.95

MAX_NUM_SEQS = 1

# vLLM GPT-OSS Blackwell/Hopper recipe commonly uses 8192.
# If startup OOM happens, first try 4096, then 2048, then 1024.
MAX_NUM_BATCHED_TOKENS = 8192

# Helpful for long context memory on GPT-OSS.
KV_CACHE_DTYPE = "fp8"

# Recommended in vLLM GPT-OSS Blackwell/Hopper recipe.
ENABLE_PREFIX_CACHING = False

MAX_CUDAGRAPH_CAPTURE_SIZE = 2048

# Critical fix for this Colab / Blackwell / CUDA 13 run:
# Disable only the FlashInfer top-k/top-p sampler.
# This keeps the rest of vLLM available, but avoids the FlashInfer sampler JIT crash.
USE_FLASHINFER_SAMPLER = False

# Do not disable all FlashInfer paths unless the server still crashes.
# Keep this False for the first retry.
DISABLE_FLASHINFER_COMPLETELY = False

# Optional hard fallback. Keep None for the first retry.
# If the server still crashes inside FlashInfer attention, set this to "TRITON_ATTN".
ATTENTION_BACKEND = None

# Add pip-installed NVIDIA CUDA libraries to LD_LIBRARY_PATH for subprocesses.
# This helps with optional libraries such as libnvrtc.so.13 when they exist in site-packages.
ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH = True

SERVER_LOG_PATH = Path("/content/vllm_gpt_oss_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gpt_oss_answer_server.pid")

LOCAL_RUNTIME_DIR = Path("/content/final_project_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence" / DATASET
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
DRIVE_IDEA_DIR = DRIVE_PROJECT_DIR / "idea_1"

DRIVE_EVIDENCE_PATH = (
    DRIVE_IDEA_DIR
    / "evidence"
    / DATASET
    / "hotpotqa_dev_2017wiki_1000_traversal_evidence.json"
)

LOCAL_EVIDENCE_PATH = LOCAL_EVIDENCE_DIR / DRIVE_EVIDENCE_PATH.name

DRIVE_ANSWER_DIR = DRIVE_IDEA_DIR / "answers" / DATASET
DRIVE_ANSWER_PATH = DRIVE_ANSWER_DIR / "hotpotqa_answer_gpt_oss.json"

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None  # None means all records.

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# This is the total completion budget for reasoning + final JSON.
# The final answer is short, but GPT-OSS can use this space for reasoning.
ANSWER_MAX_TOKENS = 32768

# Larger budget only for retry after invalid / broken JSON.
ANSWER_RETRY_MAX_TOKENS = 81920

LOW_COMPLETION_TOKENS_THRESHOLD = 2048

print("Model:", LLM_MODEL_NAME)
print("Reasoning effort:", REASONING_EFFORT)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Max num batched tokens:", MAX_NUM_BATCHED_TOKENS)
print("KV cache dtype:", KV_CACHE_DTYPE)
print("Prefix caching enabled:", ENABLE_PREFIX_CACHING)
print("Use FlashInfer sampler:", USE_FLASHINFER_SAMPLER)
print("Disable FlashInfer completely:", DISABLE_FLASHINFER_COMPLETELY)
print("Attention backend override:", ATTENTION_BACKEND)
print("Drive evidence path:", DRIVE_EVIDENCE_PATH)
print("Local evidence path:", LOCAL_EVIDENCE_PATH)
print("Answer output path:", DRIVE_ANSWER_PATH)

Model: openai/gpt-oss-120b
Reasoning effort: high
Max model len: 131072
GPU memory utilization: 0.95
Max num batched tokens: 8192
KV cache dtype: fp8
Prefix caching enabled: False
Use FlashInfer sampler: False
Disable FlashInfer completely: False
Attention backend override: None
Drive evidence path: /content/drive/MyDrive/final_project/idea_1/evidence/hotpotqa/hotpotqa_dev_2017wiki_1000_traversal_evidence.json
Local evidence path: /content/final_project_copy/evidence/hotpotqa/hotpotqa_dev_2017wiki_1000_traversal_evidence.json
Answer output path: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json


In [3]:
# cell 3
# Mount Google Drive and copy evidence to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)

if not DRIVE_EVIDENCE_PATH.exists():
    raise FileNotFoundError(f"Evidence file not found: {DRIVE_EVIDENCE_PATH}")

def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether local copy is complete.
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with temp file to avoid partial local copies.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print("Local evidence copy already exists.")
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

copy_file_to_local(DRIVE_EVIDENCE_PATH, LOCAL_EVIDENCE_PATH)

print("Evidence copied to local disk.")
print("Local evidence size MB:", LOCAL_EVIDENCE_PATH.stat().st_size / (1024 ** 2))
print("Answer directory:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Evidence copied to local disk.
Local evidence size MB: 13.081527709960938
Answer directory: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa


In [4]:
# cell 4
# Load traversal evidence.

with open(LOCAL_EVIDENCE_PATH, "r", encoding="utf-8") as f:
    evidence_records = json.load(f)

if not isinstance(evidence_records, list):
    raise RuntimeError("Evidence JSON must be a list of records.")

required_keys = ["type", "question", "answer", "evidence_chunk", "evidence_path"]

for i, rec in enumerate(evidence_records[:5]):
    missing = [k for k in required_keys if k not in rec]
    if missing:
        print(f"Warning: record {i} missing keys:", missing)

print("Number of evidence records:", len(evidence_records))
print("First source_index:", evidence_records[0].get("source_index", 0))
print("First question:", evidence_records[0].get("question"))
print("First GT answer:", evidence_records[0].get("answer"))
print("First chunks:", len(evidence_records[0].get("evidence_chunk", [])))
print("First paths:", len(evidence_records[0].get("evidence_path", [])))

Number of evidence records: 1000
First source_index: 0
First question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
First GT answer: Chief of Protocol
First chunks: 6
First paths: 5


In [5]:
# cell 5
# Start GPT-OSS vLLM server.

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_nvidia_library_dirs():
    # Find CUDA shared library directories installed by pip packages.
    dirs = []

    for base in site.getsitepackages():
        pattern = os.path.join(base, "nvidia", "*", "lib")
        for d in glob.glob(pattern):
            if os.path.isdir(d):
                dirs.append(d)

    # Remove duplicates while preserving order.
    unique_dirs = []
    seen = set()
    for d in dirs:
        if d not in seen:
            unique_dirs.append(d)
            seen.add(d)

    return unique_dirs

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--generation-config", "vllm",
    "--trust-remote-code",

    # Let GPT-OSS / MXFP4 use the correct automatic dtype.
    "--dtype", "auto",
]

if KV_CACHE_DTYPE:
    cmd.extend(["--kv-cache-dtype", KV_CACHE_DTYPE])

if MAX_CUDAGRAPH_CAPTURE_SIZE:
    cmd.extend(["--max-cudagraph-capture-size", str(MAX_CUDAGRAPH_CAPTURE_SIZE)])

if not ENABLE_PREFIX_CACHING:
    cmd.append("--no-enable-prefix-caching")
else:
    cmd.append("--enable-prefix-caching")

server_env = os.environ.copy()

# Blackwell / CUDA 13 environment consistency.
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

# Critical fix:
# Disable the FlashInfer top-k/top-p sampler that crashed during vLLM warmup.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "1" if USE_FLASHINFER_SAMPLER else "0"

# Optional fallback:
# Use this only if the server still crashes inside FlashInfer after disabling the sampler.
if DISABLE_FLASHINFER_COMPLETELY:
    server_env["VLLM_DISABLE_FLASHINFER"] = "1"

if ATTENTION_BACKEND:
    server_env["VLLM_ATTENTION_BACKEND"] = ATTENTION_BACKEND

# Add pip-installed NVIDIA library paths to help optional CUDA libraries resolve.
if ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH:
    nvidia_lib_dirs = find_nvidia_library_dirs()
    old_ld_path = server_env.get("LD_LIBRARY_PATH", "")
    merged_ld_path = ":".join(nvidia_lib_dirs + ([old_ld_path] if old_ld_path else []))
    server_env["LD_LIBRARY_PATH"] = merged_ld_path

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in [
    "VLLM_MAIN_CUDA_VERSION",
    "TORCH_CUDA_ARCH_LIST",
    "VLLM_USE_FLASHINFER_SAMPLER",
    "VLLM_DISABLE_FLASHINFER",
    "VLLM_ATTENTION_BACKEND",
    "LD_LIBRARY_PATH",
]:
    value = server_env.get(k)
    if k == "LD_LIBRARY_PATH" and value:
        print(f"{k}={value[:500]}{'...' if len(value) > 500 else ''}")
    else:
        print(f"{k}={value}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve openai/gpt-oss-120b --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.95 --max-num-seqs 1 --max-num-batched-tokens 8192 --generation-config vllm --trust-remote-code --dtype auto --kv-cache-dtype fp8 --max-cudagraph-capture-size 2048 --no-enable-prefix-caching

Important environment variables:
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_DISABLE_FLASHINFER=None
VLLM_ATTENTION_BACKEND=None
LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cupti/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/p..

In [6]:
# cell 6
# Wait for vLLM server and create OpenAI-compatible client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# OpenAI-recommended GPT-OSS sampling is temperature=1.0 and top_p=1.0.
LLM_SAMPLING_KWARGS = {
    "temperature": 1.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

# Reasoning effort is passed to the GPT-OSS chat template.
# This does not change your QA prompt; it only tells GPT-OSS how much reasoning effort to use.
LLM_EXTRA_BODY = {
    "chat_template_kwargs": {
        "reasoning_effort": REASONING_EFFORT,
    },
    "include_reasoning": True,
}

print("OpenAI-compatible client is ready.")
print("Reasoning effort:", REASONING_EFFORT)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)
print("Extra body:", LLM_EXTRA_BODY)

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(APIServer pid=2399) INFO 06-01 09:01:40 [vllm.py:984] Asynchronous scheduling is enabled.
(APIServer pid=2399) INFO 06-01 09:01:40 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=3017) INFO 06-01 09:01:52 [core.py:112] Initializing a V1 LLM engine (v0.22.1rc1.dev31+g0910f7e0e) with config: model='openai/gpt-oss-120b', speculative_config=None, tokenizer='openai/gpt-oss-120b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=gpt_oss_mxfp4, quantization_config=No

In [7]:
# cell 7
# Answer schema and prompt.

ANSWER_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "response": {
            "type": "string",
            "minLength": 1
        }
    },
    "required": ["response"]
}

ANSWER_SYSTEM_PROMPT = """
You are a careful and highly capable multi-hop open-domain question answering agent.

You will receive:
1. A HotpotQA-style question.
2. Final evidence chunks retrieved by a knowledge-graph traversal agent.
3. Final evidence paths retrieved from the same knowledge graph.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.

- Edge types:
  1. entity --relation--> entity
     - Directed edge from one entity to another entity.
     - The relation text is a short natural-language sentence describing the connection.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Traversal summary:
- A traversal agent explored the knowledge graph to collect evidence for the question.
- The selected chunks may contain bridge evidence, comparison evidence, answer-bearing evidence, or irrelevant distractors.
- The selected paths may help connect entities across multiple hops.
- Some chunks and paths can be noisy or misleading. Treat them as possible distractors, not guaranteed evidence.
- Prefer explicit evidence from chunks, but use paths to guide multi-hop reasoning when they help connect facts.

Your task:
Answer the question using only the provided evidence chunks and evidence paths.

Reasoning instructions:
- In your thinking phase, carefully analyze the question, evidence chunks, and evidence paths.
- Identify the relevant entities and the required reasoning type: bridge, comparison, or another multi-hop pattern.
- Connect facts across multiple chunks when needed.
- Use evidence paths as supporting signals for entity connections, but do not treat a path as stronger than explicit chunk text.
- Ignore distractor chunks or paths that are not needed for the answer.
- Do not use external knowledge.
- Do not guess beyond the provided evidence.

Answering rules:
1. Your primary goal is to synthesize the answer from the provided evidence.
2. Do not give up easily. Many questions require combining evidence from two or more chunks.
3. Only if the provided evidence is truly insufficient, contradictory, or does not support any answer after careful multi-hop reasoning, the final response must be exactly:
   Information not available
4. The final answer should be extremely concise.
5. Prefer a short answer phrase of 1 to 10 words when possible.
6. Never exceed 18 words unless the exact required response is Information not available.
7. Do not include explanations, citations, reasoning traces, or extra text in the final answer string.

Output format:
- You may reason during the model's thinking phase.
- After reasoning is complete, the final assistant content must contain exactly one valid JSON object.
- The JSON object must match this schema:
{
  "response": "..."
}
- Do not wrap the JSON in markdown.
- Do not add any text before or after the JSON.
""".strip()

In [8]:
# cell 8
# Prompt formatting helpers.

def clean_text_for_prompt(text):
    # Preserve text content, only normalize Python None.
    if text is None:
        return ""
    return str(text)

def format_evidence_chunks(chunks):
    # Format all chunks without truncation.
    if not chunks:
        return "No evidence chunks provided."

    blocks = []

    for i, chunk in enumerate(chunks, start=1):
        title = clean_text_for_prompt(chunk.get("title", ""))
        text = clean_text_for_prompt(chunk.get("text", ""))

        block = (
            f"[Chunk {i}]\n"
            f"Title: {title}\n"
            f"Text:\n{text}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)

def format_evidence_paths(paths):
    # Format all paths as p1, p2, ...
    if not paths:
        return "No evidence paths provided."

    lines = []

    for i, path_item in enumerate(paths, start=1):
        path_text = clean_text_for_prompt(path_item.get("path", ""))
        lines.append(f"p{i}: {path_text}")

    return "\n".join(lines)

def build_answer_prompt(record):
    # Build final QA prompt.
    question = clean_text_for_prompt(record.get("question", ""))

    chunks_text = format_evidence_chunks(record.get("evidence_chunk", []))
    paths_text = format_evidence_paths(record.get("evidence_path", []))

    prompt = f"""
Question:
{question}

Evidence chunks:
{chunks_text}

Evidence paths:
{paths_text}

Now answer the question. You may reason in the thinking phase, then output the final answer as one valid JSON object only.
""".strip()

    return prompt

# Preview one prompt.
preview_prompt = build_answer_prompt(evidence_records[0])
print(preview_prompt[:4000])
print("\nPrompt preview length in characters:", len(preview_prompt))

Question:
What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?

Evidence chunks:
[Chunk 1]
Title: Kiss and Tell (1945 film)
Text:
Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer. In the film, two teenage girls cause their respective parents much concern when they start to become interested in boys. The parents' bickering about which girl is the worse influence causes more problems than it solves.
The movie was based on the Broadway play "Kiss and Tell", which was based on the Corliss Archer short stories. The stories, play and movie were all written by F. Hugh Herbert. A sequel film, "A Kiss for Corliss", was released in 1949 and also starred Temple, but was not written by Herbert.
To boost sales and attract customers at the local bazaar, fifteen-year-old Corliss Archer and seventeen-year-old Mildred Pringle decide to start selling kisses.
When their booth at a USO bazaar fails to 

In [9]:
# cell 9
# JSON parsing and answer cleanup.

def strip_code_fence(text):
    # Remove markdown code fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Remove raw thinking blocks if returned.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def extract_json_object(text):
    # Extract first JSON object from model output.
    text = strip_code_fence(strip_think_blocks(text))
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found. Raw text:\n{text[:2000]}")

    return text[start:end + 1]

def normalize_answer_text(answer):
    # Keep answer short and clean.
    answer = strip_think_blocks(answer)
    answer = strip_code_fence(answer)
    answer = re.sub(r"\s+", " ", answer).strip()

    if not answer:
        return "Information not available"

    # Remove accidental surrounding quotes.
    if len(answer) >= 2 and answer[0] == answer[-1] and answer[0] in ['"', "'"]:
        answer = answer[1:-1].strip()

    # Enforce exact fallback text.
    if answer.lower() in {
        "not available",
        "information unavailable",
        "unknown",
        "cannot determine",
        "not enough information",
        "insufficient information",
    }:
        return "Information not available"

    return answer

def parse_answer_json(content):
    # Parse answer JSON and validate schema.
    json_text = extract_json_object(content)
    parsed = json.loads(json_text)
    validate(instance=parsed, schema=ANSWER_SCHEMA)

    response = normalize_answer_text(parsed["response"])
    parsed["response"] = response

    return parsed

In [10]:
# cell 10
# LLM answer function with retry only for invalid or broken JSON.

def usage_to_dict(usage):
    if usage is None:
        return None

    try:
        return usage.model_dump()
    except Exception:
        try:
            return dict(usage)
        except Exception:
            return None

def get_completion_tokens_from_usage(usage):
    # Read completion token count if vLLM/OpenAI-compatible usage provides it.
    usage_dict = usage_to_dict(usage)
    if not usage_dict:
        return None
    return usage_dict.get("completion_tokens")

def get_reasoning_tokens_from_usage(usage):
    # Some reasoning-compatible servers expose reasoning token counts here.
    usage_dict = usage_to_dict(usage)
    if not usage_dict:
        return None

    details = usage_dict.get("completion_tokens_details") or {}
    if isinstance(details, dict):
        return details.get("reasoning_tokens")

    return None

def get_message_content_and_reasoning(msg):
    # GPT-OSS/vLLM may expose reasoning under reasoning_content or reasoning.
    content = getattr(msg, "content", None) or ""

    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict):
                parts.append(part.get("text") or part.get("content") or "")
            else:
                parts.append(str(part))
        content = "".join(parts)

    reasoning = (
        getattr(msg, "reasoning_content", None)
        or getattr(msg, "reasoning", None)
    )

    if reasoning is None:
        try:
            msg_dict = msg.model_dump()
            reasoning = (
                msg_dict.get("reasoning_content")
                or msg_dict.get("reasoning")
            )
        except Exception:
            reasoning = None

    return content, reasoning

def should_retry_after_parse_error(response, content):
    # Retry only when the final output is not valid JSON.
    usage = response.usage
    completion_tokens = get_completion_tokens_from_usage(usage)

    finish_reason = None
    try:
        finish_reason = response.choices[0].finish_reason
    except Exception:
        pass

    low_completion = (
        completion_tokens is not None
        and completion_tokens < LOW_COMPLETION_TOKENS_THRESHOLD
    )

    hit_length_limit = finish_reason == "length"

    return {
        "retry": True,
        "completion_tokens": completion_tokens,
        "low_completion": low_completion,
        "finish_reason": finish_reason,
        "hit_length_limit": hit_length_limit,
    }

def call_llm_answer(prompt, max_retries=2):
    # Call reasoning-enabled GPT-OSS and parse short JSON answer.
    # Retry only if JSON parsing/validation fails.
    # Do not retry valid Information not available responses.
    last_error = None
    last_content = None
    last_reasoning = None
    last_retry_info = None

    messages = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    for attempt in range(max_retries + 1):
        current_messages = messages

        # First attempt uses normal budget. Retries use larger reasoning/output budget.
        current_max_tokens = ANSWER_MAX_TOKENS if attempt == 0 else ANSWER_RETRY_MAX_TOKENS

        if attempt > 0:
            repair_prompt = (
                prompt
                + "\n\nYour previous response could not be parsed as valid JSON. "
                + "You may reason again in the thinking phase with a larger output budget if needed. "
                + "After reasoning is complete, your final assistant content must contain exactly one valid JSON object like {\"response\": \"...\"}. "
                + "Do not wrap the JSON in markdown. Do not add explanations, citations, reasoning traces, or extra text outside the JSON.\n\n"
                + f"Validation error:\n{last_error}\n\n"
                + f"Previous retry info:\n{json.dumps(last_retry_info, ensure_ascii=False)}\n\n"
                + f"Previous response:\n{last_content}"
            )

            current_messages = [
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": repair_prompt},
            ]

        response = client.chat.completions.create(
            model=LLM_MODEL_NAME,
            messages=current_messages,
            max_tokens=current_max_tokens,
            **LLM_SAMPLING_KWARGS,
            extra_body=LLM_EXTRA_BODY,
        )

        msg = response.choices[0].message
        content, reasoning = get_message_content_and_reasoning(msg)

        last_content = content
        last_reasoning = reasoning

        usage_dict = usage_to_dict(response.usage)
        completion_tokens = get_completion_tokens_from_usage(response.usage)
        reasoning_tokens = get_reasoning_tokens_from_usage(response.usage)

        try:
            parsed = parse_answer_json(content)

            # Important:
            # If JSON is valid, return immediately.
            # This includes valid {"response": "Information not available"}.
            return {
                "response": parsed["response"],
                "raw_content": content,
                "reasoning_content": reasoning,
                "usage": usage_dict,
                "completion_tokens": completion_tokens,
                "reasoning_tokens": reasoning_tokens,
                "max_tokens_used": current_max_tokens,
                "attempt": attempt,
                "retry_reason": None,
            }

        except Exception as e:
            last_error = repr(e)
            last_retry_info = should_retry_after_parse_error(response, content)

            if attempt >= max_retries:
                break

            continue

    # Last-resort fallback from raw content after all JSON retries failed.
    fallback = normalize_answer_text(strip_think_blocks(last_content or ""))

    # Avoid storing invalid verbose text as answer.
    if "{" in fallback or "}" in fallback or len(fallback.split()) > 30:
        fallback = "Information not available"

    return {
        "response": fallback,
        "raw_content": last_content,
        "reasoning_content": last_reasoning,
        "usage": None,
        "parse_error": last_error,
        "retry_info": last_retry_info,
        "max_tokens_used": ANSWER_RETRY_MAX_TOKENS,
        "attempt": "fallback",
        "retry_reason": "invalid_json_after_retries",
    }

In [11]:
# cell 11
# Output I/O and resume helpers.

def atomic_write_json(path, data):
    # Atomic JSON write.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_answer_map(path):
    # Load existing answers for resume.
    path = Path(path)

    if not path.exists():
        return {}

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

    if not isinstance(data, list):
        return {}

    out = {}

    for pos, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue

        source_index = rec.get("source_index", pos)

        try:
            source_index = int(source_index)
        except Exception:
            source_index = pos

        out[source_index] = rec

    return out

def save_answer_map(path, answer_map):
    # Save answers sorted by source_index.
    ordered = [
        answer_map[idx]
        for idx in sorted(answer_map.keys())
    ]
    atomic_write_json(path, ordered)

def make_base_output_record(record):
    # Convert phase-2 record to phase-3 output schema.
    return {
        "source_index": int(record.get("source_index", 0)),
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
    }

def make_success_output_record(record, llm_result):
    # Build final answer record.
    out = make_base_output_record(record)
    out["response"] = llm_result["response"]
    out["attempt"] = llm_result.get("attempt")
    out["max_tokens_used"] = llm_result.get("max_tokens_used")
    out["completion_tokens"] = llm_result.get("completion_tokens")
    out["reasoning_tokens"] = llm_result.get("reasoning_tokens")
    out["retry_reason"] = llm_result.get("retry_reason")
    return out

def make_error_output_record(record, error):
    # Build error record but keep response field.
    out = make_base_output_record(record)
    out["response"] = "Information not available"
    out["error"] = str(error)
    out["traceback"] = traceback.format_exc()
    return out

existing_answers = load_existing_answer_map(DRIVE_ANSWER_PATH)

print("Existing answer records:", len(existing_answers))
print("Answer output:", DRIVE_ANSWER_PATH)

Existing answer records: 0
Answer output: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json


In [12]:
# cell 12
# Run phase 3 answer generation and save after each question.

start_idx = int(ANSWER_START_INDEX)
end_idx = len(evidence_records) if ANSWER_END_INDEX is None else int(ANSWER_END_INDEX)

run_records = evidence_records[start_idx:end_idx]

print("Total evidence records:", len(evidence_records))
print("Run range:", start_idx, "to", end_idx)
print("Existing answers:", len(existing_answers))
print("Output:", DRIVE_ANSWER_PATH)

progress = tqdm(run_records, desc="Generating answers", dynamic_ncols=True)

for local_pos, record in enumerate(progress, start=1):
    source_index = int(record.get("source_index", start_idx + local_pos - 1))

    # Skip completed records.
    if source_index in existing_answers:
        old = existing_answers[source_index]
        if isinstance(old, dict) and old.get("response") and "error" not in old:
            progress.set_postfix({
                "source_index": source_index,
                "status": "skipped",
                "saved": len(existing_answers),
            })
            continue

    try:
        prompt = build_answer_prompt(record)
        llm_result = call_llm_answer(prompt)

        output_record = make_success_output_record(record, llm_result)
        existing_answers[source_index] = output_record
        status = "ok"

    except Exception as e:
        output_record = make_error_output_record(record, e)
        existing_answers[source_index] = output_record
        status = "error"

    if local_pos % SAVE_EVERY_N == 0:
        save_answer_map(DRIVE_ANSWER_PATH, existing_answers)

    if local_pos % CLEAR_CACHE_EVERY_N == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    progress.set_postfix({
        "source_index": source_index,
        "status": status,
        "saved": len(existing_answers),
    })

save_answer_map(DRIVE_ANSWER_PATH, existing_answers)

print("Answer generation finished.")
print("Saved records:", len(existing_answers))
print("Output file:", DRIVE_ANSWER_PATH)

Total evidence records: 1000
Run range: 0 to 1000
Existing answers: 0
Output: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json


Generating answers:   0%|          | 0/1000 [00:00<?, ?it/s]

Answer generation finished.
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json


In [13]:
# cell 13
# Inspect saved answers.

with open(DRIVE_ANSWER_PATH, "r", encoding="utf-8") as f:
    saved_answers = json.load(f)

print("Saved answers:", len(saved_answers))
print("Output path:", DRIVE_ANSWER_PATH)

for rec in saved_answers[:5]:
    print("=" * 100)
    print("source_index:", rec.get("source_index"))
    print("type:", rec.get("type"))
    print("question:", rec.get("question"))
    print("gt:", rec.get("gt"))
    print("response:", rec.get("response"))

Saved answers: 1000
Output path: /content/drive/MyDrive/final_project/idea_1/answers/hotpotqa/hotpotqa_answer_gpt_oss.json
source_index: 0
type: bridge
question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
gt: Chief of Protocol
response: Chief of Protocol of the United States
source_index: 1
type: bridge
question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
gt: Animorphs
response: Animorphs
source_index: 2
type: bridge
question: The director of the romantic comedy "Big Stone Gap" is based in what New York city?
gt: Greenwich Village, New York City
response: New York City
source_index: 3
type: bridge
question: 2014 S/S is the debut album of a South Korean boy group that was formed by who?
gt: YG Entertainment
response: YG Entertainment
source_index: 4
type: bridge
question: Who was known by his stage name Aladin and helped